# LLM and RAG — Hands-On Tutorial

**Duration:** ~1 hour  
**Stack:** HuggingFace Transformers · Gemma 3 4B · Sentence Transformers · ChromaDB

## What we will build

```
User Question
     |
     v
[ Embedding Model ]  -->  [ ChromaDB Vector Store ]  -->  Top-K Chunks
                                                               |
                                                               v
                                                      [ Cross-Encoder Re-ranker ]
                                                               |
                                                               v
                                                        Best Chunks
                                                               |
                                                               v
                                               [ Gemma 3 4B ] + Context  -->  Answer
```

## Sections
1. Setup & Dependencies
2. LLM Basics — Loading and Querying Gemma
3. Why We Need RAG — Showing LLM Limitations
4. Document Loading — Wikipedia Article
5. Chunking Strategies
6. Embeddings — Turning Text into Vectors
7. ChromaDB — Storing Vectors
8. Retrieval — Finding Relevant Chunks
9. Re-ranking — Improving Retrieval Quality
10. Complete RAG Pipeline

## Section 1 — Setup

Install all required packages. This only needs to run once.

In [ ]:
!pip install -q transformers accelerate torch chromadb sentence-transformers wikipedia scikit-learn

### HuggingFace Authentication

Gemma is a gated model. You need to:
1. Create a free account at https://huggingface.co
2. Accept the Gemma license at https://huggingface.co/google/gemma-3-4b-it
3. Generate an access token at https://huggingface.co/settings/tokens
4. Paste it when prompted below

In [ ]:
from huggingface_hub import login
login()  # Paste your HuggingFace token when prompted

---
## Section 2 — LLM Basics

### What is an LLM?

A Large Language Model (LLM) is a neural network trained on massive amounts of text.  
It learns to predict the next token (word/subword) given previous tokens.

```
Input:  "The capital of France is"
Output: "Paris"          <-- model predicts the most likely next token
```

**Gemma 3 4B** is Google's open-weight LLM with 4 billion parameters.  
We use the `-it` (instruction-tuned) variant, which is fine-tuned to follow instructions.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "google/gemma-3-4b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (may take a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  # Use half precision to save memory
    device_map="auto"            # Automatically use GPU if available, else CPU
)
print(f"Model loaded on: {model.device}")
print(f"Model parameters: ~4 billion")

### What is a Tokenizer?

LLMs don't read text — they read **tokens** (integer IDs).  
The tokenizer converts text ↔ tokens.

In [ ]:
# Let's see how the tokenizer works
sample_text = "Artificial intelligence is transforming the world."
tokens = tokenizer.encode(sample_text)
token_words = [tokenizer.decode([t]) for t in tokens]

print(f"Original text : {sample_text}")
print(f"Token IDs     : {tokens}")
print(f"Token count   : {len(tokens)}")
print(f"Tokens decoded: {token_words}")
print()
print("Notice: words can split into multiple tokens (e.g. 'transform' + 'ing')")

In [ ]:
# Helper function to generate a response from Gemma
def generate(prompt, max_new_tokens=256, system_prompt=None):
    if system_prompt:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt}
        ]
    else:
        messages = [{"role": "user", "content": prompt}]

    # apply_chat_template returns BatchEncoding (dict-like) in newer transformers,
    # or a plain tensor in older versions — handle both.
    tokenized = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )
    if hasattr(tokenized, "input_ids"):
        input_ids = tokenized.input_ids.to(model.device)
    else:
        input_ids = tokenized.to(model.device)

    input_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy decoding — deterministic output
        )

    # Decode only the NEW tokens (not the input prompt)
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Quick test
response = generate("What is machine learning? Answer in 2 sentences.")
print(response)

---
## Section 3 — Why We Need RAG

LLMs have two major limitations:

| Problem | Description |
|---|---|
| **Knowledge cutoff** | Trained on data up to a certain date — no knowledge of recent events |
| **Hallucination** | Confidently generates plausible-sounding but incorrect facts |
| **No source citations** | Cannot tell you WHERE an answer came from |

Let's demonstrate hallucination and knowledge limits:

In [ ]:
# Ask about something very specific — the model may hallucinate or admit ignorance
questions = [
    "Who won the 2025 Nobel Prize in Physics and what was it for?",
    "What is the exact enrollment count at the University of Hyderabad in 2025?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, max_new_tokens=150)}")
    print("-" * 60)

### RAG fixes this

**Retrieval-Augmented Generation (RAG)** gives the LLM access to a **knowledge base** at query time.

```
Without RAG:  Question  -->  LLM  -->  Answer (from training data only)

With RAG:     Question  -->  Retriever  -->  Relevant Chunks
                                                  +
                             LLM  -->  Answer (grounded in retrieved facts)
```

The LLM acts as a **reasoning engine**, not a **knowledge store**.

---
## Section 4 — Loading Documents

Our knowledge base will be the Wikipedia article on **Artificial Intelligence**.  
This is rich with facts the model should be able to retrieve and cite.

In [ ]:
import wikipedia

topic = "Artificial intelligence"
print(f"Fetching Wikipedia article: '{topic}'...")

page = wikipedia.page(topic, auto_suggest=False)
raw_text = page.content

print(f"Title      : {page.title}")
print(f"URL        : {page.url}")
print(f"Characters : {len(raw_text):,}")
print(f"Words      : {len(raw_text.split()):,}")
print()
print("--- First 800 characters ---")
print(raw_text[:800])

---
## Section 5 — Chunking Strategies

We cannot feed the entire Wikipedia article into the LLM at once —  
LLMs have a **context window limit** (max tokens they can process).

Solution: Split the document into smaller **chunks** and retrieve only the relevant ones.

### Strategy 1: Fixed-Size Chunking (Naive)

In [ ]:
def fixed_size_chunk(text, chunk_size=500):
    """Split text every N characters — simple but crude."""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i : i + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

naive_chunks = fixed_size_chunk(raw_text, chunk_size=500)
print(f"Total chunks: {len(naive_chunks)}")
print()
print("--- End of chunk 0 ---")
print(repr(naive_chunks[0][-120:]))
print()
print("--- Start of chunk 1 ---")
print(repr(naive_chunks[1][:120]))
print()
print("Problem: chunks cut in the middle of words/sentences!")

### Strategy 2: Fixed-Size with Overlap

Add overlap so context is not lost at chunk boundaries.

In [ ]:
def chunk_with_overlap(text, chunk_size=500, overlap=100):
    """Split with a sliding window so adjacent chunks share some text."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap  # Slide forward (less than full chunk_size)
    return chunks

overlap_chunks = chunk_with_overlap(raw_text, chunk_size=500, overlap=100)
print(f"Total chunks: {len(overlap_chunks)}")
print()
print("--- End of chunk 0 ---")
print(repr(overlap_chunks[0][-100:]))
print()
print("--- Start of chunk 1 (notice overlap with chunk 0 above) ---")
print(repr(overlap_chunks[1][:100]))

### Strategy 3: Sentence-Aware Chunking (Best for RAG)

Respect sentence boundaries — each chunk contains complete sentences.

In [ ]:
import re

def sentence_chunk(text, max_chars=600, overlap_sentences=2):
    """Group complete sentences into chunks, with sentence-level overlap."""
    # Split on sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_sentences = []
    current_len = 0

    for sentence in sentences:
        if current_len + len(sentence) > max_chars and current_sentences:
            chunks.append(" ".join(current_sentences))
            # Keep last N sentences for context continuity
            current_sentences = current_sentences[-overlap_sentences:]
            current_len = sum(len(s) for s in current_sentences)
        current_sentences.append(sentence)
        current_len += len(sentence)

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

sentence_chunks = sentence_chunk(raw_text, max_chars=600, overlap_sentences=2)

print("Chunking strategy comparison:")
print(f"  Fixed-size (no overlap):  {len(naive_chunks)} chunks")
print(f"  Fixed-size + overlap:     {len(overlap_chunks)} chunks")
print(f"  Sentence-aware:           {len(sentence_chunks)} chunks")
print()
print("--- Sample sentence chunk ---")
print(sentence_chunks[5])

In [ ]:
# We'll use sentence chunks for the rest of the notebook
chunks = sentence_chunks
print(f"Using {len(chunks)} sentence-aware chunks")
print(f"Average chunk length: {sum(len(c) for c in chunks) // len(chunks)} characters")

---
## Section 6 — Embeddings

### What is an Embedding?

An **embedding** is a list of numbers (a vector) that captures the *meaning* of text.  
Similar text → similar vectors.

```
"The dog barked"   --> [0.12, -0.34, 0.88, ...]  (384 numbers)
"A puppy howled"   --> [0.11, -0.31, 0.85, ...]  (very close!)
"Interest rates"   --> [-0.45, 0.72, -0.23, ...] (very different)
```

We use `all-MiniLM-L6-v2` — small (80MB), fast, and high quality.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Semantic similarity demo
sentences = [
    "Artificial intelligence involves creating intelligent machines.",
    "AI systems are designed to mimic human cognitive functions.",
    "The stock market closed higher on Tuesday.",
]

embeddings = embed_model.encode(sentences)
sim_matrix = cosine_similarity(embeddings)

labels = ["AI sentence 1", "AI sentence 2", "Stock market"]
print("Cosine similarity (1.0 = identical meaning, 0.0 = unrelated):")
print()
header = f"{'':>15}  " + "  ".join(f"{l:>13}" for l in labels)
print(header)
for i, label in enumerate(labels):
    row = f"{label:>15}  " + "  ".join(f"{sim_matrix[i][j]:>13.4f}" for j in range(len(labels)))
    print(row)

print()
print("AI sentences are highly similar; stock market sentence is unrelated.")

---
## Section 7 — ChromaDB Vector Store

A **vector store** is a database optimised for storing and searching embeddings.  
Instead of exact keyword search, it does **semantic (meaning-based) search**.

**ChromaDB** is a lightweight, in-process vector database — perfect for learning.

In [ ]:
import chromadb

# Custom embedding function that wraps our sentence transformer
class SentenceTransformerEF(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return embed_model.encode(input).tolist()

# Create an in-memory ChromaDB client
chroma_client = chromadb.Client()

# A "collection" in ChromaDB is like a table — holds documents + their embeddings
collection = chroma_client.create_collection(
    name="ai_wikipedia",
    embedding_function=SentenceTransformerEF(),
    metadata={"hnsw:space": "cosine"}  # Use cosine distance for similarity
)
print("Collection created!")

In [ ]:
# Add all chunks to ChromaDB
# ChromaDB will automatically embed them using our embedding function
BATCH_SIZE = 50
print(f"Adding {len(chunks)} chunks to ChromaDB...")

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    collection.add(
        documents=batch,
        ids=[f"chunk_{j}" for j in range(i, i + len(batch))]
    )
    print(f"  Added {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks", end="\r")

print(f"\nDone! Collection contains {collection.count()} chunks.")

---
## Section 8 — Retrieval

Given a user question, we embed it and find the most **semantically similar chunks** in ChromaDB.  
This is called **dense retrieval** (vector search).

In [ ]:
def retrieve(query, n_results=5):
    """Find the n most relevant chunks for a query."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    docs      = results["documents"][0]
    distances = results["distances"][0]   # Lower distance = more similar
    return docs, distances


query = "What are the main subfields of artificial intelligence?"
docs, distances = retrieve(query, n_results=4)

print(f"Query: {query}\n")
for i, (doc, dist) in enumerate(zip(docs, distances)):
    print(f"--- Result {i+1}  (cosine distance: {dist:.4f}) ---")
    print(doc[:300])
    print()

In [ ]:
# Try a few more queries to see retrieval in action
test_queries = [
    "History of artificial intelligence",
    "Ethics and risks of AI",
    "Machine learning algorithms",
]

for q in test_queries:
    docs, dists = retrieve(q, n_results=1)
    print(f"Q: {q}")
    print(f"Best match (dist={dists[0]:.4f}):")
    print(docs[0][:250])
    print("-" * 60)

---
## Section 9 — Re-ranking

### The Problem with Dense Retrieval Alone

Dense retrieval encodes query and documents **separately** — it's fast but approximate.  
It can miss subtle relevance or surface chunks that are topically similar but not actually useful.

### Solution: Cross-Encoder Re-ranker

A **cross-encoder** reads the query and a document **together** and outputs a relevance score.  
It's slower (can't pre-compute) but much more accurate.

```
Bi-encoder (retrieval):    embed(query) vs embed(doc)   --> fast, approximate
Cross-encoder (re-rank):   model(query + doc)           --> slow, accurate

Strategy: retrieve 10-20 candidates fast, then re-rank to get top 3 accurately.
```

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading cross-encoder re-ranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Re-ranker loaded!")


def rerank(query, docs, top_k=3):
    """Score each (query, doc) pair and return top_k by score."""
    pairs  = [(query, doc) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

In [ ]:
# Compare: retrieval order vs re-ranked order
query = "What are the ethical concerns about artificial intelligence?"

# Step 1: Retrieve 10 candidates
candidates, distances = retrieve(query, n_results=10)

print("=" * 60)
print("BEFORE re-ranking (top 3 by embedding distance):")
print("=" * 60)
for i, (doc, dist) in enumerate(zip(candidates[:3], distances[:3])):
    print(f"\nRank {i+1}  (distance: {dist:.4f})")
    print(doc[:250])

# Step 2: Re-rank with cross-encoder
reranked = rerank(query, candidates, top_k=3)

print()
print("=" * 60)
print("AFTER re-ranking (top 3 by cross-encoder score):")
print("=" * 60)
for i, (doc, score) in enumerate(reranked):
    print(f"\nRank {i+1}  (score: {score:.4f})")
    print(doc[:250])

---
## Section 10 — Complete RAG Pipeline

Now we assemble all the pieces into one function.

```
Question
  --> [Embed]            -- encode the question
  --> [ChromaDB Search]  -- retrieve 10 candidate chunks
  --> [Re-rank]          -- pick top 3 most relevant
  --> [Build Prompt]     -- inject chunks as context
  --> [Gemma 3 4B]       -- generate grounded answer
```

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant that answers questions based strictly on the provided context.
If the answer is not in the context, say "I don't have enough information to answer that based on the provided context."
Do not make up information. Cite relevant facts from the context in your answer."""


def rag_pipeline(question, n_retrieve=10, n_rerank=3, verbose=True):
    if verbose:
        print(f"Question: {question}\n")

    # --- Step 1: Retrieve candidates ---
    if verbose:
        print(f"[1/3] Retrieving {n_retrieve} candidate chunks...")
    candidates, _ = retrieve(question, n_results=n_retrieve)

    # --- Step 2: Re-rank ---
    if verbose:
        print(f"[2/3] Re-ranking to select top {n_rerank} chunks...")
    reranked   = rerank(question, candidates, top_k=n_rerank)
    top_chunks = [doc for doc, _ in reranked]

    # --- Step 3: Build prompt with context ---
    context = "\n\n---\n\n".join(top_chunks)
    prompt  = f"Context:\n{context}\n\nQuestion: {question}"

    # --- Step 4: Generate answer ---
    if verbose:
        print("[3/3] Generating answer with Gemma 3 4B...\n")
    answer = generate(prompt, max_new_tokens=512, system_prompt=SYSTEM_PROMPT)

    if verbose:
        print("=" * 60)
        print("ANSWER")
        print("=" * 60)
        print(answer)
        print("=" * 60)

    return answer, top_chunks

In [ ]:
# Test the full pipeline
answer, sources = rag_pipeline(
    "What are the main approaches and subfields of artificial intelligence?"
)

In [ ]:
# Inspect the source chunks used
print("Source chunks used to generate the answer:\n")
for i, chunk in enumerate(sources):
    print(f"--- Source {i+1} ---")
    print(chunk[:300])
    print()

In [ ]:
# Side-by-side comparison: with RAG vs without RAG
question = "What role does natural language processing play in artificial intelligence?"

print("WITHOUT RAG (LLM only):")
print("-" * 60)
no_rag = generate(question, max_new_tokens=200)
print(no_rag)

print()
print("WITH RAG (retrieval + re-ranking + LLM):")
print("-" * 60)
with_rag, _ = rag_pipeline(question, verbose=False)
print(with_rag)

In [ ]:
# Try your own questions!
your_question = "What are the risks of artificial general intelligence?"

answer, _ = rag_pipeline(your_question)

---
## Summary

### What We Covered

| Concept | What it does |
|---|---|
| **Tokenizer** | Converts text into integer IDs the LLM can process |
| **LLM (Gemma 3 4B)** | Generates fluent text; limited to training-time knowledge |
| **RAG** | Augments LLM with retrieved facts at query time |
| **Chunking** | Splits documents into retrievable pieces; overlap preserves context |
| **Embeddings** | Dense vector representations of meaning; similar text → similar vectors |
| **ChromaDB** | Vector store for fast semantic search over embeddings |
| **Dense Retrieval** | Embed query, find nearest chunks by cosine similarity |
| **Cross-Encoder Re-ranking** | Score each (query, chunk) pair together for precise ranking |

### Key Takeaways

1. LLMs hallucinate — always ground answers in retrieved sources for factual tasks.
2. Chunking strategy matters — sentence-aware chunking with overlap beats naive fixed-size splits.
3. Retrieve more, re-rank fewer — get 10 candidates fast, then pick 3 accurately.
4. The LLM is the **reasoner**, not the **knowledge store** — keep them separate.

### Next Steps

- **Hybrid Search**: combine keyword search (BM25) with dense retrieval for better coverage
- **HyDE**: generate a hypothetical answer first, then use it as the retrieval query
- **Persistent ChromaDB**: save the vector store to disk so you don't re-embed every run
- **Agentic RAG**: let the LLM decide when and what to retrieve